# Vortex Gemma 4 E4B LoRA/QLoRA Colab

Security: uses only `/MyDrive/VortexTraining/`; no Gmail, Photos, Contacts, full Drive scan, web datasets, token printing, infinite training, or auto-promotion.

Base model: `google/gemma-4-E4B-it`. Output: LoRA/QLoRA adapter only.

In [ ]:
BASE_MODEL = "google/gemma-4-E4B-it"
DRIVE_ROOT = "/content/drive/MyDrive/VortexTraining"
TRAIN_JSONL = f"{DRIVE_ROOT}/datasets/processed/gemma4_flutter_python_sft.jsonl"
EVAL_JSONL = f"{DRIVE_ROOT}/datasets/eval/gemma4_flutter_python_eval.jsonl"
ADAPTER_DIR = f"{DRIVE_ROOT}/outputs/adapters/gemma4_flutter_python_lora"
LOG_DIR = f"{DRIVE_ROOT}/outputs/logs/gemma4_flutter_python_lora"
REPORT_DIR = f"{DRIVE_ROOT}/outputs/reports"

TRAIN_CFG = {
    "max_seq_length": 1024,
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "max_steps": 200,
    "load_in_4bit": True,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
}


In [ ]:
!pip -q install -U "transformers>=5.5,<5.6" "peft>=0.10" "trl" "datasets" "accelerate" "bitsandbytes" "safetensors" "huggingface_hub" "kernels>=0.11.1"
# Optional. If install/import/model load fails, fallback path below is used.
!pip -q install -U unsloth || true


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
for sub in [
    'datasets/raw', 'datasets/processed', 'datasets/eval', 'notebooks',
    'outputs/adapters', 'outputs/logs', 'outputs/reports', 'exports'
]:
    Path(DRIVE_ROOT, sub).mkdir(parents=True, exist_ok=True)

assert Path(TRAIN_JSONL).exists(), TRAIN_JSONL
assert Path(EVAL_JSONL).exists(), EVAL_JSONL


In [ ]:
from huggingface_hub import notebook_login

# Paste token into widget. Token is not printed.
notebook_login()


In [ ]:
import hashlib, json, re
from collections import Counter

secret_re = re.compile(r'(?i)(api[_-]?key|access[_-]?token|auth[_-]?token|secret|password|cookie|authorization)\\s*[:=]')
domain_re = {
    'flutter': re.compile(r'(?i)flutter|widget|buildcontext|scaffold|layout|overflow'),
    'dart': re.compile(r'(?i)\\bdart\\b|future<|stream<|async|await|state management'),
    'python': re.compile(r'(?i)\\bpython\\b|def\\s+\\w+\\(|pytest|typing|pydantic'),
    'fastapi': re.compile(r'(?i)fastapi|APIRouter|@app\\.|Depends\\(|HTTPException'),
    'vortex': re.compile(r'(?i)vortex|c3rnt2|hf_train|adapter|rtx4080|gemma'),
}

def audit_jsonl(path):
    seen = set(); domains = Counter(); invalid = 0; secrets = 0; rows = 0; dups = 0
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        rows += 1
        obj = json.loads(line)
        txt = json.dumps(obj, ensure_ascii=False)
        h = hashlib.sha256(txt.encode()).hexdigest()
        if h in seen: dups += 1
        seen.add(h)
        if secret_re.search(txt): secrets += 1
        msgs = obj.get('messages')
        roles = [m.get('role') for m in msgs] if isinstance(msgs, list) else []
        if roles != ['system', 'user', 'assistant']: invalid += 1
        domain = obj.get('domain') or 'unknown'
        domains[domain] += 1
    return {'rows': rows, 'invalid': invalid, 'duplicates': dups, 'secret_hits': secrets, 'domains': dict(domains)}

train_audit = audit_jsonl(TRAIN_JSONL)
eval_audit = audit_jsonl(EVAL_JSONL)
min_domains = {'flutter': 12, 'dart': 8, 'python': 40, 'fastapi': 8, 'vortex': 40}
gate_reasons = []
if train_audit['rows'] < 180: gate_reasons.append('too_small')
if train_audit['invalid']: gate_reasons.append('invalid_chat_format')
if train_audit['duplicates']: gate_reasons.append('duplicates_present')
if train_audit['secret_hits']: gate_reasons.append('secret_like_text_present')
for k, v in min_domains.items():
    if train_audit['domains'].get(k, 0) < v:
        gate_reasons.append(f'domain_{k}_underrepresented')
gate_ok = not gate_reasons
quality_report = {'gate_ok': gate_ok, 'gate_reasons': gate_reasons, 'train': train_audit, 'eval': eval_audit}
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
Path(REPORT_DIR, 'dataset_quality_report_colab.json').write_text(json.dumps(quality_report, indent=2), encoding='utf-8')
print(json.dumps(quality_report, indent=2))
if not gate_ok:
    raise SystemExit('Dataset quality gate failed. Do not train yet.')


In [ ]:
import torch
from datasets import load_dataset

bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
dtype = torch.bfloat16 if bf16 else torch.float16
print({'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'bf16': bf16})

raw = load_dataset('json', data_files={'train': TRAIN_JSONL, 'eval': EVAL_JSONL})


In [ ]:
def format_messages(tokenizer, row):
    messages = row['messages']
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        parts = []
        for m in messages:
            parts.append(f"{m['role'].upper()}: {m['content']}")
        return '\n\n'.join(parts)

def resolve_lora_targets(model, requested):
    supported = []
    for name, module in model.named_modules():
        cls = module.__class__.__name__.lower()
        if 'linear' not in cls and 'clippable' not in cls:
            continue
        leaf = name.split('.')[-1]
        if leaf in requested and leaf not in supported:
            supported.append(leaf)
    return supported or ['q_proj', 'k_proj', 'v_proj', 'o_proj']


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

compat = {'unsloth_import': False, 'unsloth_load': False, 'path': 'transformers_peft'}
model = tokenizer = None

try:
    from unsloth import FastLanguageModel
    compat['unsloth_import'] = True
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=TRAIN_CFG['max_seq_length'],
        dtype=dtype,
        load_in_4bit=TRAIN_CFG['load_in_4bit'],
    )
    targets = resolve_lora_targets(model, TRAIN_CFG['target_modules'])
    model = FastLanguageModel.get_peft_model(
        model,
        r=TRAIN_CFG['lora_rank'],
        target_modules=targets,
        lora_alpha=TRAIN_CFG['lora_alpha'],
        lora_dropout=TRAIN_CFG['lora_dropout'],
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=3407,
    )
    compat['unsloth_load'] = True
    compat['path'] = 'unsloth'
except Exception as exc:
    compat['unsloth_error'] = type(exc).__name__ + ': ' + str(exc)[:500]
    bnb = BitsAndBytesConfig(
        load_in_4bit=TRAIN_CFG['load_in_4bit'],
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        quantization_config=bnb,
        device_map='auto',
        attn_implementation='sdpa',
    )
    model = prepare_model_for_kbit_training(model)
    targets = resolve_lora_targets(model, TRAIN_CFG['target_modules'])
    model = get_peft_model(model, LoraConfig(
        r=TRAIN_CFG['lora_rank'],
        lora_alpha=TRAIN_CFG['lora_alpha'],
        lora_dropout=TRAIN_CFG['lora_dropout'],
        target_modules=targets,
        bias='none',
        task_type='CAUSAL_LM',
    ))

Path(REPORT_DIR, 'compatibility.json').write_text(json.dumps(compat, indent=2), encoding='utf-8')
print(json.dumps(compat, indent=2))
print('LoRA targets:', targets)


In [ ]:
def to_text(row):
    return {'text': format_messages(tokenizer, row)}

text_ds = raw.map(to_text, remove_columns=raw['train'].column_names)

def tok(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=TRAIN_CFG['max_seq_length'])
    enc['labels'] = enc['input_ids'].copy()
    return enc

tok_ds = text_ds.map(tok, batched=True, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir=LOG_DIR,
    per_device_train_batch_size=TRAIN_CFG['per_device_train_batch_size'],
    gradient_accumulation_steps=TRAIN_CFG['gradient_accumulation_steps'],
    learning_rate=TRAIN_CFG['learning_rate'],
    max_steps=TRAIN_CFG['max_steps'],
    logging_steps=5,
    save_steps=50,
    eval_steps=50,
    eval_strategy='steps',
    bf16=bf16,
    fp16=not bf16,
    report_to='none',
    optim='paged_adamw_8bit',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tok_ds['train'],
    eval_dataset=tok_ds['eval'],
    data_collator=collator,
)
train_result = trainer.train()
metrics = trainer.evaluate()
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
Path(ADAPTER_DIR, 'training_args.json').write_text(args.to_json_string(), encoding='utf-8')
Path(REPORT_DIR, 'train_metrics.json').write_text(json.dumps({'train': train_result.metrics, 'eval': metrics}, indent=2), encoding='utf-8')
metrics


In [ ]:
from transformers import pipeline

eval_prompts = [
    "Fix a Flutter layout overflow in a Row with a long title and trailing button.",
    "Design a responsive Flutter settings screen for desktop and phone.",
    "Explain basic Dart state management for a chat UI.",
    "Create a FastAPI endpoint that returns adapter status with typed response.",
    "Improve Python typing for a function that parses JSONL samples.",
    "Refactor a bad function that reads files, validates data, and writes output.",
    "Explain the Vortex Gemma 4 adapter registry and promotion flow.",
    "Write a precise Codex prompt to add a focused Vortex backend test.",
]

model.eval()
rows = []
for prompt in eval_prompts:
    messages = [{'role': 'system', 'content': 'You are Vortex, a direct coding assistant.'}, {'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) if hasattr(tokenizer, 'apply_chat_template') else prompt
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=220, do_sample=False)
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    rows.append({'prompt': prompt, 'output': decoded})

Path(REPORT_DIR, 'eval_outputs.jsonl').write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
md = ['# Eval Report', '', f'Base model: `{BASE_MODEL}`', f'Adapter: `{ADAPTER_DIR}`', f'Training path: `{compat["path"]}`', '', '## Metrics', '```json', json.dumps(metrics, indent=2), '```', '', '## Outputs']
for r in rows:
    md += ['', f'### {r["prompt"]}', r['output']]
Path(REPORT_DIR, 'eval_report.md').write_text('\n'.join(md) + '\n', encoding='utf-8')
print('\n'.join(md[:20]))


In [ ]:
readme = f"""# Vortex Gemma 4 Flutter/Python LoRA

- Base model: `{BASE_MODEL}`
- Dataset: `{TRAIN_JSONL}`
- Eval dataset: `{EVAL_JSONL}`
- Method: {compat['path']} LoRA/QLoRA
- Max steps: {TRAIN_CFG['max_steps']}
- Max seq length: {TRAIN_CFG['max_seq_length']}
- LoRA rank: {TRAIN_CFG['lora_rank']}
- LoRA alpha: {TRAIN_CFG['lora_alpha']}
- LoRA dropout: {TRAIN_CFG['lora_dropout']}

## Limits

Small adapter, not full base-model training. Quality depends on reviewed dataset coverage. Do not promote without local Vortex doctor/bench.

## Import into Vortex

Copy this directory to `c3_rnt2_ai/data/registry/hf_train/gemma4_e4b/colab_flutter_python_lora/`, then run local eval before promotion.
"""
Path(ADAPTER_DIR, 'README.md').write_text(readme, encoding='utf-8')
print('Adapter saved:', ADAPTER_DIR)
